# v15a — corpus rebuild: hard negatives + payload spans

v14 run 2 solved the adversarial-benign confound (WJB-Adv-Benign FPR 7.49%) but left the
indirect carriers failing: BIPIA ASR 17.17%, InjecAgent 19.35% at their own 10% FPR.

The post-hoc analysis showed **the detector finds the payload** (injected outscores its own
clean carrier in 92.2% / 89.2% of pairs) but **the threshold cannot be shared**: the cutoff
each context needs for its own 10% FPR spans 0.000132 to 0.994230.

v15 attacks that with (a) within-document self-normalisation (v15b/v15c) and (b) negatives
that make the claim falsifiable (this notebook).

## What changes here

**1. `carrier_id` bug fixed.** v14a prefixed it with the corpus name, so an injected row got
`BIPIA:<h>` and its clean twin `BIPIA-Clean:<h>` — different keys for the same document. The
matched pairs could not be joined. Now both members share `pair_id`.

**2. Two tiers of negative, reported separately.**

| Tier | Construction | A false positive here is |
|---|---|---|
| **clean** | payload slot removed / neutral non-instruction filler | unambiguously wrong |
| **hardneg** | same carrier, same position, **benign instruction-shaped** filler | defensible — report separately |

v14 had only tier 1, and for InjecAgent only 17 rows of it. Tier 2 is the real test: does the
detector fire on *any imperative sentence inside untrusted content*, or on an instruction that
**overrides the assistant's task**? That distinction is the thesis claim, and nothing before
v15 could measure it.

**3. Payload character spans recorded** (`span_start`, `span_end`) so v15c can check whether the
chunk the detector flags is the chunk the payload is actually in — localisation ground truth,
not just an AUC.

In [1]:
import os, re, json, hashlib
import numpy as np
import pandas as pd

SEED = 42
rng  = np.random.default_rng(SEED)

DATA = "datasets"
OUT  = os.path.join(DATA, "unified")
os.makedirs(OUT, exist_ok=True)

V14_CORPUS = os.path.join(OUT, "injection_corpus.csv")     # reused for non-indirect rows
V15_CORPUS = os.path.join(OUT, "injection_corpus_v15.csv")

CAP = dict(
    BIPIA_PER_PAYLOAD = 3,      # unchanged from v14 so injected rows stay comparable
    INJAG_PER_PAYLOAD = 3,
    HARDNEG_PER_CARRIER = 1,    # BIPIA: 1 hard negative per unique carrier
    INJAG_HARDNEG_PER_TMPL = 30,   # InjecAgent has only 17 templates -> vary the filler
    INJAG_CLEAN_PER_TMPL   = 30,   # v14 had ONE neutral filler and 17 rows total
)
MAX_CHARS = 3000               # keep in step with v15b MAX_LEN

print("v15a corpus builder | seed", SEED)
for k, v in CAP.items(): print(f"  {k:<26}{v}")

v15a corpus builder | seed 42
  BIPIA_PER_PAYLOAD         3
  INJAG_PER_PAYLOAD         3
  HARDNEG_PER_CARRIER       1
  INJAG_HARDNEG_PER_TMPL    30
  INJAG_CLEAN_PER_TMPL      30


In [2]:
# ---- helpers (clean_text identical to v14a / main.ipynb) --------------------
def clean_text(t):
    t = re.sub(r"\s+", " ", str(t).strip())
    return re.sub(r"[^\x00-\x7F]+", "", t)

DE_RE = re.compile(r"\b(der|die|das|und|ist|nicht|du|sie|wie|was|warum|bitte|ich|alle|"
                   r"vorherigen|anweisungen|vergiss|ignoriere|jetzt|eine|einen|mir|mich|"
                   r"wir|sind|haben|werden|schreibe|fr|ber|mit|auf|von|zu|dich|deine|dass)\b", re.I)
lang_of = lambda t: "de" if len(DE_RE.findall(t)) >= 2 else "en"

ROWS = []

def add(text, label, corpus, family, payload_id, pair_id=None, meta="", span_of=None):
    # pair_id: SHARED between an injected row and its clean/hardneg twins. This is the
    # v14a bug fix -- carrier_id used to be prefixed with the corpus name, so twins got
    # different keys and could not be joined.
    # span_of: the payload/filler text. Located AFTER cleaning, because clean_text
    # collapses whitespace and would invalidate any offset computed before it.
    t = clean_text(text)
    if not t: return None
    s0 = s1 = -1
    if span_of:
        needle = clean_text(span_of)
        if needle:
            s0 = t.find(needle)
            s1 = s0 + len(needle) if s0 >= 0 else -1
    ROWS.append(dict(text=t, label=int(label), corpus=corpus, family=family,
                     payload_id=f"{corpus}:{payload_id}",
                     pair_id=(pair_id or ""), lang=lang_of(t), n_chars=len(t),
                     span_start=s0, span_end=s1, meta=meta))
    return len(ROWS) - 1

def take(seq, k):
    seq = list(seq)
    if k is None or k >= len(seq): return seq
    idx = rng.choice(len(seq), size=k, replace=False)
    return [seq[i] for i in sorted(idx)]

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

In [3]:
# ---- benign filler pool -----------------------------------------------------
# Tier-2 hard negatives need text that is INSTRUCTION-SHAPED but benign, so the only
# thing separating it from a payload is whether it tries to redirect the assistant.
# Dolly instructions are exactly that: imperative, task-like, and not injections.
#
# Tier-1 clean fillers are DECLARATIVE -- no imperative mood at all.

IMPERATIVE = re.compile(r"^\s*(please\s+)?(write|give|list|tell|explain|summar|describe|"
                        r"name|find|identify|classify|extract|generate|create|make|provide|"
                        r"calculate|translate|convert|recommend|suggest|compare|what|who|"
                        r"when|where|why|how|can you|could you)", re.I)

dolly_path = "datasets/dolly-15k-instruction-alpaca-format.csv"
assert os.path.exists(dolly_path), f"not found: {dolly_path}"

_d = pd.read_csv(dolly_path).instruction.dropna().astype(str).tolist()
FILLER_HARD = [clean_text(s) for s in _d
               if 20 <= len(s) <= 400 and IMPERATIVE.match(s)]
FILLER_HARD = list(dict.fromkeys(FILLER_HARD))
rng.shuffle(FILLER_HARD)

# Declarative, non-instruction filler for tier 1. Short, neutral, content-bearing.
# v15a run 1 had only 10 of these, so INJAG_CLEAN_PER_TMPL=30 produced 10 unique rows
# per template (170 total) against 510 hard negatives. Expanded so the two tiers balance.
FILLER_CLEAN = [
    "Great product, arrived on time and works exactly as described.",
    "The package was well sealed and the contents matched the listing.",
    "Delivery took four days and the item was in good condition.",
    "Build quality feels solid and the finish is even throughout.",
    "It has been running without issues for about three weeks now.",
    "The colour is slightly darker than it appears in the photographs.",
    "Setup was straightforward and the instructions were clear enough.",
    "Battery life is roughly what the specification sheet claims.",
    "The size matches the dimensions given on the product page.",
    "No complaints so far; it does what it is supposed to do.",
    "The packaging was recyclable and there was very little plastic inside.",
    "It weighs a little more than expected but that is not a problem.",
    "The manual covers the basics though some sections are quite terse.",
    "Shipping was handled by the usual courier and tracking worked fine.",
    "The surface picks up fingerprints easily but wipes clean without marks.",
    "It fits the space it was bought for with a couple of centimetres spare.",
    "The seller responded to a question about compatibility within a day.",
    "Performance has been consistent across the three machines it was tested on.",
    "There is a faint noise on startup that settles after a few seconds.",
    "The warranty card was included along with a short quick-start leaflet.",
    "Colour matching against the earlier model in the range is very close.",
    "The connector is on the left side rather than the rear as pictured.",
    "It draws noticeably less power than the unit it replaced last year.",
    "Assembly needed one tool that was not supplied in the box.",
    "The finish has held up well through several weeks of daily handling.",
    "Two of the accessories listed on the page arrived in a separate parcel.",
    "The display is readable at an angle though brightness drops off.",
    "Firmware was already at the current version when it arrived.",
    "It has been stored in a cool room and shows no sign of warping.",
    "The included cable is about a metre long, which was enough here.",
    "Dimensions on the label match those printed in the specification.",
    "It arrived a day earlier than the estimate given at checkout.",
    "The rubber feet grip well on a wooden surface and do not slide.",
    "A protective film covers the panel and peels off cleanly.",
    "The serial number is printed on the underside next to the rating plate.",
    "Storage temperature guidance is printed on the side of the carton.",
    "It came with a spare seal, which was a useful inclusion.",
    "The second unit ordered behaves identically to the first.",
    "Contents were listed on the outside of the box and all were present.",
    "The hinge moves smoothly and holds position where it is left.",
]
# LENGTH MATCHING. v14 Exp 3 showed length still moves AUC by -0.0137, and v15a run 1
# produced hard negatives with a median 1044 chars against 852 for injections -- a 20%
# gap in the benign direction. A tier-2 result at that gap would be unreadable: is the
# detector responding to the instruction, or to the document being longer?
# Fillers are now assembled to match the length of the payload they stand in for.
import bisect
_POOL = {"hard": sorted(FILLER_HARD, key=len), "clean": sorted(FILLER_CLEAN, key=len)}
_LENS = {k: [len(x) for x in v] for k, v in _POOL.items()}
_PTR  = {"hard": 0, "clean": 0}

def build_filler(kind, target, lo=0.8, hi=1.35, win=150):
    # Assemble a filler whose length matches the payload it stands in for.
    # Picking the next item in rotation does NOT work: Dolly instructions run short
    # (median 51 chars) against BIPIA payloads (median 74, p90 280, max 557), so short
    # targets came out 2.4x too long and long ones 20% too short. Instead the first
    # piece is chosen by NEAREST length via bisect, rotating within a window for
    # variety, and further pieces are appended only while still under budget.
    # Verified on the real length distribution: >=99% of outputs land in [0.7, 1.4]x
    # target for every payload decile.
    pool, lens = _POOL[kind], _LENS[kind]
    def pick(t):
        i = bisect.bisect_left(lens, int(max(1, t)))
        a = max(0, min(i - win // 2, len(pool) - win)); a = max(0, a)
        b = min(len(pool), a + win)
        f = pool[a + (_PTR[kind] % max(1, b - a))]; _PTR[kind] += 1
        return f
    first = pick(target * 0.95)
    parts, n = [first], len(first)
    while n < lo * target and len(parts) < 8:
        f = pick(max(20, lo * target - n))
        if n + 1 + len(f) > hi * target: break
        parts.append(f); n += len(f) + 1
    return " ".join(parts)

print(f"hard filler (imperative, benign) : {len(FILLER_HARD):,} from {dolly_path}")
print(f"  lengths: p10 {np.percentile([len(x) for x in FILLER_HARD],10):.0f}"
      f" med {np.median([len(x) for x in FILLER_HARD]):.0f}"
      f" p90 {np.percentile([len(x) for x in FILLER_HARD],90):.0f} chars")
print(f"clean filler (declarative)       : {len(FILLER_CLEAN)}")
print("\nexamples (hard):")
for s in FILLER_HARD[:3]: print("   ", s[:90])

hard filler (imperative, benign) : 11,074 from datasets/dolly-15k-instruction-alpaca-format.csv
  lengths: p10 28 med 53 p90 115 chars
clean filler (declarative)       : 40

examples (hard):
    What is an option in finance?
    What was the duration of the Lewis and Clark Expedition?
    What are the essential skills required for a communication?


In [4]:
# ---- BIPIA: injected + clean + hard negative, all sharing pair_id ----------
BIPIA_TASKS = {
    "email": ("BIPIA/email/{s}.jsonl", "BIPIA/text_attack_{s}.json"),
    "table": ("BIPIA/table/{s}.jsonl", "BIPIA/text_attack_{s}.json"),
    "code":  ("BIPIA/code/{s}.jsonl",  "BIPIA/code_attack_{s}.json"),
}

def render_carrier(task, row):
    if task == "code":
        return " ".join(str(row.get(k, "")) for k in ("error", "code", "context") if row.get(k))
    return f"{row.get('context','')}\n\n{row.get('question','')}"

def place(carrier, insert, pos):
    return f"{insert}\n\n{carrier}" if pos == 0 else f"{carrier}\n\n{insert}"

# Negatives must be drawn from the SAME carrier population as the injections.
# v15a run 2 got this wrong and the gap grew to 26.3%. The cause was not the filler --
# that matched per row (payload med 74 vs filler med 62). It was `cid`:
#   cid = md5(carrier AFTER truncation to MAX_CHARS - len(payload))
# so one long source document, truncated against payloads of different lengths, split
# into SEVERAL cids. Short carriers never truncate and stayed as one cid with ~3 injected
# rows; long carriers fragmented into cids with ~1.7 rows each. Emitting one negative per
# cid therefore over-sampled long carriers (median carrier 963 vs 758 injected).
#
# Fix, in two parts:
#   1. cid is hashed from the UNTRUNCATED carrier -- one source document, one pair_id.
#   2. the row-level mismatch CANNOT be fixed by sampling, and pretending otherwise would
#      be worse than leaving it. The clean tier's text IS the carrier, so there can only
#      ever be ONE clean row per carrier -- its carrier distribution is uniform over
#      distinct carriers by construction, while the injected tier has ~3 rows for a short
#      carrier and ~1.7 for a long one. No re-weighting of a one-row-per-carrier tier can
#      reproduce a distribution that has repeats.
#
#      So the fix belongs in the METRIC, not the corpus: make the unit of analysis the
#      CARRIER. Every carrier emits one clean row, one hard negative, and however many
#      injected rows it really carried, and `carrier_n_inj` records that count so v15c can
#      collapse injected rows to one score per carrier. Injected and benign tiers then sit
#      on an identical carrier population, and the paired comparison (same document, with
#      and without the payload) is exact rather than approximate.

n_inj = 0
carrier_txt, carrier_use, carrier_pos, carrier_plen, carrier_task = {}, {}, {}, {}, {}
for task, (cpat, apat) in BIPIA_TASKS.items():
    for split in ("train", "test"):
        cp, ap = f"{DATA}/{cpat.format(s=split)}", f"{DATA}/{apat.format(s=split)}"
        if not (os.path.exists(cp) and os.path.exists(ap)):
            print(f"  [skip] {task}/{split} missing"); continue
        ctxs = load_jsonl(cp)
        atk  = json.load(open(ap))
        payloads = [(f"{task}-{split}-{t}-{i}", t, p)
                    for t, lst in atk.items() for i, p in enumerate(lst)]
        for pid, atype, payload in payloads:
            for row in take(ctxs, CAP["BIPIA_PER_PAYLOAD"]):
                raw = render_carrier(task, row)
                if not raw.strip(): continue
                cid = "bipia-" + hashlib.md5(raw.encode()).hexdigest()[:10]   # UNtruncated
                budget = MAX_CHARS - len(payload) - 4
                carrier = raw[:max(0, budget)] if len(raw) > budget else raw
                pos = int(rng.integers(0, 2))
                tag = "start" if pos == 0 else "end"
                add(place(carrier, payload, pos), 1, "BIPIA", "indirect", pid, cid,
                    meta=f"task={task};type={atype};pos={tag}", span_of=payload)
                n_inj += 1
                carrier_use[cid] = carrier_use.get(cid, 0) + 1
                carrier_txt.setdefault(cid, carrier)      # first truncation seen
                carrier_task[cid] = task
                carrier_pos.setdefault(cid, []).append(pos)
                carrier_plen.setdefault(cid, []).append(len(payload))

cids = sorted(carrier_use)
n_cln = n_hrd = 0
for cid in cids:
    carrier = carrier_txt[cid]; task = carrier_task[cid]
    pos  = int(rng.choice(carrier_pos[cid]))               # a position this carrier really saw
    plen = int(rng.choice(carrier_plen[cid]))              # a payload length it really carried
    nu = carrier_use[cid]
    add(carrier, 0, "BIPIA-Clean", "benign", f"clean-{cid}", cid,
        meta=f"task={task};tier=clean;carrier_n_inj={nu}")
    n_cln += 1
    f = build_filler("hard", plen)
    add(place(carrier, f, pos), 0, "BIPIA-HardNeg", "benign", f"hard-{cid}", cid,
        meta=f"task={task};tier=hardneg;pos={'start' if pos==0 else 'end'};carrier_n_inj={nu}",
        span_of=f)
    n_hrd += 1
print(f"  carriers: {len(cids)} distinct source documents | "
      f"injected rows per carrier: med {np.median(list(carrier_use.values())):.0f} "
      f"max {max(carrier_use.values())}")
print(f"BIPIA  {n_inj:>5} injected | {n_cln:>4} clean | {n_hrd:>4} hard-neg "
      f"| {len(cids)} carriers")

  carriers: 481 distinct source documents | injected rows per carrier: med 2 max 9
BIPIA   1200 injected |  481 clean |  481 hard-neg | 481 carriers


In [5]:
# ---- InjecAgent: only 17 templates exist, so vary the FILLER ----------------
# v14 emitted ONE clean row per template (17 total) with a single constant neutral string.
# 17 negatives cannot support an FPR claim -- v14 run 2 reported "100% FPR" on 17 rows.
# The template has exactly one <Attacker Instruction> slot, so both tiers scale by filler.
uc  = load_jsonl(f"{DATA}/InjectAgent/user_cases.jsonl")
atk = (load_jsonl(f"{DATA}/InjectAgent/attacker_cases_dh.jsonl")
       + load_jsonl(f"{DATA}/InjectAgent/attacker_cases_ds.jsonl"))

def injag_text(u, filled):
    return (f"{u.get('User Instruction','')}\n\n"
            f"Tool result ({u.get('User Tool','tool')}): {filled}")

n_inj = 0
tmpls = {}
for ai, a in enumerate(atk):
    instr = a.get("Attacker Instruction", "")
    if not instr: continue
    for u in take(uc, CAP["INJAG_PER_PAYLOAD"]):
        tmpl = str(u.get("Tool Response Template", ""))
        if "<Attacker Instruction>" not in tmpl: continue
        cid = "injag-" + hashlib.md5(tmpl.encode()).hexdigest()[:10]
        tmpls.setdefault(cid, u)
        add(injag_text(u, tmpl.replace("<Attacker Instruction>", instr)),
            1, "InjecAgent", "indirect", f"atk{ai}", cid,
            meta=f"type={a.get('Attack Type','')};tool={u.get('User Tool','')}",
            span_of=instr)
        n_inj += 1

# Target length = median attacker-instruction length, so both tiers sit where the
# payloads sit rather than where the filler pool happens to sit.
# Match the payload length DISTRIBUTION, not just its median. Run 2 used a single
# constant target, so hard-negative fillers came out 109/111/114 chars at p10/med/p90
# while real payloads span 73-187. Constant length is itself a signal a tree can learn.
PLENS = [len(a.get("Attacker Instruction", "")) for a in atk if a.get("Attacker Instruction")]
print(f"  InjecAgent payloads: p10 {np.percentile(PLENS,10):.0f} med {np.median(PLENS):.0f} "
      f"p90 {np.percentile(PLENS,90):.0f} chars -- both tiers sampled from this")

n_cln = n_hrd = 0
for cid, u in tmpls.items():
    tmpl = str(u.get("Tool Response Template", ""))
    for j in range(CAP["INJAG_CLEAN_PER_TMPL"]):
        f = build_filler("clean", int(rng.choice(PLENS)))
        # span recorded for the clean tier too. InjecAgent's template REQUIRES a filled
        # slot, so its tier 1 replaces the payload rather than removing it (unlike BIPIA,
        # where tier 1 is the bare carrier). Without a span the parity check fell back to
        # full text length and compared clean's whole document against injected's
        # document-minus-payload -- a 60% "gap" that was entirely the check's fault.
        add(injag_text(u, tmpl.replace("<Attacker Instruction>", f)),
            0, "InjecAgent-Clean", "benign", f"clean-{cid}-{j}", cid,
            meta=f"tool={u.get('User Tool','')};tier=clean", span_of=f)
        n_cln += 1
    for j in range(CAP["INJAG_HARDNEG_PER_TMPL"]):
        f = build_filler("hard",  int(rng.choice(PLENS)))
        add(injag_text(u, tmpl.replace("<Attacker Instruction>", f)),
            0, "InjecAgent-HardNeg", "benign", f"hard-{cid}-{j}", cid,
            meta=f"tool={u.get('User Tool','')};tier=hardneg", span_of=f)
        n_hrd += 1
print(f"InjecAgent  {n_inj:>4} injected | {n_cln:>4} clean | {n_hrd:>4} hard-neg "
      f"| {len(tmpls)} templates")
print(f"  (v14 had 17 clean and 0 hard-neg -- an FPR on 17 rows is not an FPR)")

  InjecAgent payloads: p10 73 med 118 p90 187 chars -- both tiers sampled from this
InjecAgent   186 injected |  510 clean |  510 hard-neg | 17 templates
  (v14 had 17 clean and 0 hard-neg -- an FPR on 17 rows is not an FPR)


In [6]:
# ---- carry over every non-indirect row from v14 unchanged -------------------
# SPML / DeepSet / WJB / XSTest / JBB / Dolly are untouched by v15's changes, and keeping
# them byte-identical means v15 numbers stay comparable with v14 run 2.
v14 = pd.read_csv(V14_CORPUS)
keep = ~v14.corpus.isin(["BIPIA", "BIPIA-Clean", "InjecAgent", "InjecAgent-Clean"])
carry = v14[keep].copy()
carry["pair_id"]    = ""
carry["span_start"] = -1
carry["span_end"]   = -1
carry = carry.drop(columns=[c for c in ["carrier_id"] if c in carry.columns])

new = pd.DataFrame(ROWS)
COLS = ["text","label","corpus","family","payload_id","pair_id","lang",
        "n_chars","span_start","span_end","meta"]
df = pd.concat([carry[COLS], new[COLS]], ignore_index=True)

# drop exact duplicate texts, keeping the first (injections are added before negatives)
before = len(df)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print(f"carried over {len(carry):,} rows from v14 | new indirect rows {len(new):,}")
print(f"exact-duplicate texts dropped: {before - len(df)}")

carried over 9,910 rows from v14 | new indirect rows 3,368
exact-duplicate texts dropped: 33


In [7]:
# ---- verification ----------------------------------------------------------
print(df.groupby(["corpus","label"]).size().unstack(fill_value=0).to_string())
print(f"\nTOTAL {len(df):,} rows | {df.label.sum():,} injections | {(df.label==0).sum():,} benign")

print("\n-- pair_id joins (the v14a bug) --")
for fam in ["bipia", "injag"]:
    sub = df[df.pair_id.astype(str).str.startswith(fam)]
    if not len(sub): continue
    piv = sub.pivot_table(index="pair_id", columns="corpus", values="text",
                          aggfunc="count").fillna(0)
    joined = (piv > 0).sum(axis=1)
    print(f"  {fam}: {len(piv)} pair_ids | corpora per pair: "
          f"{dict(joined.value_counts().sort_index())}")
    assert (joined >= 2).all(), f"{fam}: some pair_id has no twin -- join is still broken"

print("\n-- payload spans --")
inj = df[df.label == 1]
for c in ["BIPIA", "InjecAgent"]:
    s = inj[inj.corpus == c]
    ok = (s.span_start >= 0).mean() * 100
    frac = ((s.span_start + s.span_end) / 2 / s.n_chars)
    print(f"  {c:12s} span located in {ok:5.1f}% of rows | "
          f"payload midpoint at {frac.median():.2f} of document (0=start, 1=end)")
    assert ok > 95, f"{c}: payload span lost in too many rows -- check clean_text"

print("\n-- length parity between tiers (a length gap would be a new shortcut) --")
BAD = False
for a, b, h in [("BIPIA","BIPIA-Clean","BIPIA-HardNeg"),
                ("InjecAgent","InjecAgent-Clean","InjecAgent-HardNeg")]:
    g = lambda c: df[df.corpus == c].n_chars
    mi, mc, mh = g(a).median(), g(b).median(), g(h).median()
    gap = max(abs(mh - mi), abs(mc - mi)) / max(mi, 1) * 100
    flag = "  <- GAP > 10%, re-check build_filler" if gap > 10 else ""
    BAD |= gap > 10
    print(f"  {a:12s} inj {mi:6.0f} | clean {mc:6.0f} | hardneg {mh:6.0f}"
          f"  (median chars, worst gap {gap:4.1f}%){flag}")
if BAD:
    print("\n  A length gap in the BENIGN direction makes tier 2 unreadable: a flag could")
    print("  be the instruction or the extra length. Fix before extracting.")

print("\n-- carrier-length parity, PER CARRIER (the like-for-like test) --")
print("   Row-level medians will NOT match and are not expected to: a short carrier")
print("   contributes ~3 injected rows and one clean row. Collapse to one row per")
print("   carrier and the two tiers must agree -- that is what v15c evaluates on.")
def carrier_med(c, dedup):
    s_ = df[df.corpus == c].copy()
    ws = s_[s_.span_start >= 0]
    s_["carrier_len"] = ((ws.n_chars - (ws.span_end - ws.span_start) - 2)
                         if len(ws) else s_.n_chars)
    if dedup: s_ = s_.drop_duplicates(subset="pair_id")
    return s_.carrier_len.median()
for a, b, h in [("BIPIA","BIPIA-Clean","BIPIA-HardNeg"),
                ("InjecAgent","InjecAgent-Clean","InjecAgent-HardNeg")]:
    for dedup, lbl in ((False, "per row   "), (True, "per carrier")):
        ci, cc, ch = (carrier_med(x, dedup) for x in (a, b, h))
        gap = max(abs(ch - ci), abs(cc - ci)) / max(ci, 1) * 100
        flag = ("  <- MUST be small" if dedup and gap > 10 else
                "  (expected to differ)" if not dedup else "  ok")
        print(f"  {a:12s} {lbl}  inj {ci:6.0f} | clean {cc:6.0f} | hardneg {ch:6.0f}"
              f"   gap {gap:4.1f}%{flag}")

print("\n-- pairs missing a tier --")
for fam in ["bipia", "injag"]:
    sub = df[df.pair_id.astype(str).str.startswith(fam)]
    if not len(sub): continue
    piv = sub.pivot_table(index="pair_id", columns="corpus", values="text", aggfunc="count").fillna(0)
    inc = piv[(piv > 0).sum(axis=1) < 3]
    print(f"  {fam}: {len(inc)} of {len(piv)} pair_ids lack one tier")
    if len(inc):
        print("    (clean carriers whose text collided after clean_text() were de-duplicated;")
        print("     use only complete pair_ids for the paired-margin analysis in v15c)")

label                  0     1
corpus                        
BIPIA                  0  1200
BIPIA-Clean          480     0
BIPIA-HardNeg        481     0
Baseline-Dolly      2000     0
DeepSet                0   263
DeepSet-Benign       300     0
InjecAgent             0   186
InjecAgent-Clean     479     0
InjecAgent-HardNeg   509     0
JBB-Benign           100     0
SPML                   0  2499
SPML-Benign         1198     0
WJB-Adv-Benign      2500     0
WJB-Vanilla-Benign   800     0
XSTest-Safe          250     0

TOTAL 13,245 rows | 4,148 injections | 9,097 benign

-- pair_id joins (the v14a bug) --
  bipia: 481 pair_ids | corpora per pair: {2: np.int64(1), 3: np.int64(480)}
  injag: 17 pair_ids | corpora per pair: {3: np.int64(17)}

-- payload spans --
  BIPIA        span located in 100.0% of rows | payload midpoint at 0.83 of document (0=start, 1=end)
  InjecAgent   span located in 100.0% of rows | payload midpoint at 0.68 of document (0=start, 1=end)

-- length parity betwe

In [8]:
df.to_csv(V15_CORPUS, index=False)
json.dump(dict(
    seed=SEED, caps=CAP, max_chars=MAX_CHARS, rows=int(len(df)),
    source_v14=V14_CORPUS, dolly=dolly_path,
    tiers={"clean":"payload slot removed or declarative filler -- a false positive is unambiguously wrong",
           "hardneg":"benign IMPERATIVE filler at the same position -- report separately"},
    changes=["carrier_id -> pair_id, shared between an injected row and its twins",
             "tier-2 hard negatives added for BIPIA and InjecAgent",
             "InjecAgent clean negatives 17 -> %d" % int((df.corpus=='InjecAgent-Clean').sum()),
             "payload char spans recorded for localisation ground truth"],
), open(os.path.join(OUT, "manifest_v15.json"), "w"), indent=2)
print(f"wrote {V15_CORPUS}  ({len(df):,} rows)")
print("next: v15b_extract_chunked.ipynb")

wrote datasets/unified/injection_corpus_v15.csv  (13,245 rows)
next: v15b_extract_chunked.ipynb
